# Bending vs LTB: Predicted and Actual Failure Strengths

For each beam in the 2-variable dataset, compute:
- Predicted bending failure load (from Ix)
- Predicted LTB failure load (from Iy, J)
- Stability ratio R = Mcr/My
- Governing predicted load = min(bending, LTB)
- Compare to actual measured failure load

Material: Sunlu PLA+ 2.0 (blue)  
Datasheet: E_flex = 2740 MPa, sigma_flex = 81.8 MPa, density = 1.21 g/cm³

In [ ]:
import numpy as np
import pandas as pd

# --- Sunlu PLA+ 2.0 datasheet (ASTM D790 / D638) ---
E = 2.74e9           # Pa, flexural modulus
SIGMA_Y = 81.8e6     # Pa, flexural strength
DENSITY = 1210       # kg/m^3
G = E / 2.6          # shear modulus, assuming nu ~ 0.3

# --- Geometry constants ---
TOTAL_H = 25.0       # mm, total beam height
B = 16.0             # mm, flange width
L = 0.2023           # m, span (3-point bending)
C1 = 1.35            # moment gradient factor for 3-point bending

In [ ]:
def calc_Ix(H_mm, b_mm, B_mm=B, total_h=TOTAL_H):
    """Strong-axis second moment of area (m^4)."""
    H = H_mm / 1e3
    b = b_mm / 1e3
    Bf = B_mm / 1e3
    h = (total_h / 1e3 - H) / 2.0
    if h <= 0:
        return 0.0
    return (b * H**3) / 12 + 2 * (Bf * h**3 / 12 + Bf * h * ((H + h) / 2)**2)

def calc_Iy(H_mm, b_mm, B_mm=B, total_h=TOTAL_H):
    """Weak-axis second moment of area (m^4)."""
    H = H_mm / 1e3
    b = b_mm / 1e3
    Bf = B_mm / 1e3
    h = (total_h / 1e3 - H) / 2.0
    if h <= 0:
        return 0.0
    return (H * b**3) / 12 + 2 * (h * Bf**3) / 12

def calc_J(H_mm, b_mm, B_mm=B, total_h=TOTAL_H):
    """Torsional constant, thin-wall approximation (m^4)."""
    H = H_mm / 1e3
    b = b_mm / 1e3
    Bf = B_mm / 1e3
    h = (total_h / 1e3 - H) / 2.0
    if h <= 0:
        return 0.0
    return (H * b**3 + 2 * Bf * h**3) / 3

def beam_calcs(H_mm, b_mm):
    h_mm = (TOTAL_H - H_mm) / 2.0
    Ix = calc_Ix(H_mm, b_mm)
    Iy = calc_Iy(H_mm, b_mm)
    J = calc_J(H_mm, b_mm)
    y_max = TOTAL_H / 2e3  # 0.0125 m

    My = SIGMA_Y * Ix / y_max
    Mcr = (C1 * np.pi / L) * np.sqrt(E * Iy * G * J) if (Iy > 0 and J > 0) else 0.0
    R = Mcr / My if My > 0 else 0.0

    P_bend = 4 * My / L
    P_ltb = 4 * Mcr / L
    P_gov = min(P_bend, P_ltb)

    return {
        'h_mm': h_mm,
        'Ix_mm4': Ix * 1e12,
        'Iy_mm4': Iy * 1e12,
        'J_mm4': J * 1e12,
        'R': R,
        'P_bend_N': P_bend,
        'P_ltb_N': P_ltb,
        'P_govern_N': P_gov,
    }

In [ ]:
df = pd.read_csv('../data/I_beam_data_2var.csv')

rows = []
for _, r in df.iterrows():
    b = r['b_web_mm']
    H = r['H_web_mm']
    actual = r['Strength N']
    c = beam_calcs(H, b)
    rows.append({
        'Beam': r['Beam Number'],
        'b': b,
        'H': H,
        'R': c['R'],
        'P_bend': c['P_bend_N'],
        'P_ltb': c['P_ltb_N'],
        'P_ltb/P_bend': c['P_ltb_N'] / c['P_bend_N'] if c['P_bend_N'] > 0 else 0,
        'P_govern': c['P_govern_N'],
        'P_actual': actual,
        'Actual/Predicted': actual / c['P_govern_N'] if c['P_govern_N'] > 0 else 0,
        'Failure mode': str(r.get('Column 1', '')),
        'Notes': str(r.get('Column 2', '')),
    })

result = pd.DataFrame(rows)

fmt = {
    'b': '{:.1f}',
    'H': '{:.1f}',
    'R': '{:.2f}',
    'P_bend': '{:.0f}',
    'P_ltb': '{:.0f}',
    'P_ltb/P_bend': '{:.2f}',
    'P_govern': '{:.0f}',
    'P_actual': '{:.0f}',
    'Actual/Predicted': '{:.2f}',
}

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 200)
result.style.format(fmt)

In [ ]:
print('\nBeams where LTB governs (R < 1):')
ltb_gov = result[result['R'] < 1.0]
if len(ltb_gov) > 0:
    print(ltb_gov[['Beam', 'b', 'H', 'R', 'P_bend', 'P_ltb', 'P_actual', 'Failure mode']].to_string(index=False))
else:
    print('  None')

print('\nBeams where bending governs but R < 1.5 (marginal):')
marginal = result[(result['R'] >= 1.0) & (result['R'] < 1.5)]
if len(marginal) > 0:
    print(marginal[['Beam', 'b', 'H', 'R', 'P_bend', 'P_ltb', 'P_actual', 'Failure mode']].to_string(index=False))
else:
    print('  None')

print(f'\nMean Actual/Predicted ratio: {result["Actual/Predicted"].mean():.2f}')
print(f'Std  Actual/Predicted ratio: {result["Actual/Predicted"].std():.2f}')